In [ ]:
import pandas as pd
import numpy as np
import re
import zipfile

zip_path = "../00_Data/original_kaggle_data_download_archive.zip"

with zipfile.ZipFile(zip_path) as z:
    print(z.namelist())  # See what files are inside
    f1 = pd.read_csv(z.open("1429_1.csv"))
    f2 = pd.read_csv(z.open("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv"))
    f3 = pd.read_csv(z.open("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"))

In [ ]:
"""
Data preparation for the Automated Customer Reviews project.
Merges the 3 raw Datafiniti/Amazon review exports, deduplicates, cleans the
corrupted `name` field, maps star ratings to sentiment classes, and writes
a single clean dataset for downstream modeling.
"""

OUT_PATH = "clean_reviews.csv"

print(f"f1 {f1.shape}, f2 {f2.shape}, f3 {f3.shape}")

# ---- Keep only columns relevant to NLP tasks ----
keep_cols = ["id", "name", "asins", "brand", "categories", "primaryCategories",
             "reviews.date", "reviews.rating", "reviews.text", "reviews.title"]

for df in (f1, f2, f3):
    for c in keep_cols:
        if c not in df.columns:
            df[c] = np.nan

merged = pd.concat([f1[keep_cols], f2[keep_cols], f3[keep_cols]], ignore_index=True)
print(f"merged raw {merged.shape}")

# ---- Deduplicate: same product + review text + rating ----
merged = merged.dropna(subset=["reviews.text", "reviews.rating"])
merged = merged.drop_duplicates(subset=["id", "reviews.text", "reviews.rating"])
print(f"after dedup {merged.shape}")
print(f"unique product IDs: {merged['id'].nunique()}")
print(f"unique product names (raw): {merged['name'].nunique()}")

# ---- Fix corrupted `name` field (two names concatenated together) ----
def clean_name(name):
    if pd.isna(name):
        return name
    name = str(name)
    # Datafiniti corruption pattern: "Name A,,,,Name B" or names glued with commas
    # Keep first product-name segment (before a run of 2+ commas, or before an
    # obvious second product name starting after a comma cluster)
    parts = re.split(r",{2,}", name)
    first = parts[0].strip()
    return first

merged["name_clean"] = merged["name"].apply(clean_name)
print(f"unique product names (cleaned): {merged['name_clean'].nunique()}")



In [ ]:
# ---- Sentiment mapping ----
def map_sentiment(rating):
    if pd.isna(rating):
        return np.nan
    rating = int(rating)
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

merged["sentiment"] = merged["reviews.rating"].apply(map_sentiment)
merged = merged.dropna(subset=["sentiment"])

# ---- Combine title + text as model input ----
merged["review_text_full"] = (
    merged["reviews.title"].fillna("") + ". " + merged["reviews.text"].fillna("")
).str.strip()
merged = merged[merged["review_text_full"].str.len() > 3]

print("\nFinal dataset:")
print(f"  rows: {len(merged)}")
print(f"  unique product IDs: {merged['id'].nunique()}")
print(f"  unique cleaned product names: {merged['name_clean'].nunique()}")
print("\nSentiment class distribution:")
print(merged["sentiment"].value_counts())
print(merged["sentiment"].value_counts(normalize=True).round(3))

merged.to_csv(OUT_PATH, index=False)
print(f"\nSaved -> {OUT_PATH}")

## Create a unique product table

In [ ]:
# ---- Clustering only the unique product names ----

products = (
    merged[["name_clean", "primaryCategories"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(products.shape)

## Vectorize the product names

In [ ]:
# ---- Using bigrams (ngram_range=(1,2)) helps distinguish phrases ----

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=1
)

X = vectorizer.fit_transform(products["clean_product_name"])

## Apply K-Means

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=20
)

products["cluster"] = kmeans.fit_predict(X)

## Inspect each cluster

In [ ]:
# ---- VALIDATE SUGGESTIONS ----

for cluster in sorted(products.cluster.unique()):

    print("="*50)
    print(f"Cluster {cluster}")

    display(
        products.loc[
            products.cluster==cluster,
            ["clean_product_name","primaryCategories"]
        ]
    )

In [ ]:
# ---- assess whether the clusters align with the known categories. ----

products[
    ["primaryCategories","cluster"]
].value_counts()

## Build a rule-based mapper

In [ ]:
# ----- create explicit mapping rules, after interpreting the clusters

def categorize_product(name, primary):

    text = str(name).lower()
    category = str(primary).lower()

    if "laptop" in text or "computer" in category:
        return "Computers"

    elif "camera" in text or "photo" in category:
        return "Cameras"

    elif any(word in text for word in
             ["cable","charger","adapter"]):
        return "Accessories"

    elif any(word in text for word in
             ["tv","speaker","monitor"]):
        return "Electronics"

    else:
        return "Other"

## Apply to the full dataset

In [ ]:
# ------ every product will get one of the five meta-categories ----

merged["meta_category"] = merged.apply(
    lambda row: categorize_product(
        row["name_clean"],
        row["primaryCategories"]
    ),
    axis=1
)

## Validate the results

In [ ]:
# ---- Check the distribution -----

merged["meta_category"].value_counts()

In [ ]:
# ----- Review examples from each category -----

for category in merged["meta_category"].unique():

    print(category)

    display(
        merged.loc[
            merged.meta_category==category,
            "clean_product_name"
        ].drop_duplicates().head(20)
    )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("clean_reviews.csv")
df = df.dropna(subset=["review_text_full", "sentiment"])
 
X = df["review_text_full"]
y = df["sentiment"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print("Train class balance:\n", y_train.value_counts(normalize=True).round(3))
 
vectorizer = TfidfVectorizer(
    max_features=30000, ngram_range=(1, 2), min_df=3, stop_words="english"
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)